# Notebook 4: Distributed Coordination — Decomposition Methods for BRP

## References

1. **Boyd, S., Parikh, N., Chu, E., Peleato, B., Eckstein, J. (2011).** *"Distributed Optimization and Statistical Learning via the Alternating Direction Method of Multipliers."* Foundations and Trends in Machine Learning, 3(1), 1–122. [DOI: 10.1561/2200000016](https://doi.org/10.1561/2200000016)

2. **Dall'Anese, E., Zhu, H., Giannakis, G. B. (2013).** *"Distributed optimal power flow for smart microgrids."* IEEE Transactions on Smart Grid, 4(3), 1464–1475. [DOI: 10.1109/TSG.2013.2248175](https://doi.org/10.1109/TSG.2013.2248175)

3. **Kraning, M., Chu, E., Lavaei, J., Boyd, S. (2014).** *"Dynamic network energy management via proximal message passing."* Foundations and Trends in Optimization, 1(2), 73–126. [DOI: 10.1561/2400000002](https://doi.org/10.1561/2400000002)

## What these papers bring

**Boyd et al. (2011)** is the foundational ADMM reference. Key insight: coupling constraints can be handled via iterative dual variable (price signal) updates, enabling distributed optimization.

**Dall'Anese et al. (2013)** apply distributed optimization to microgrid OPF via ADMM.

**Kraning et al. (2014)** develop proximal message-passing for network energy management.

## What is implemented below

We demonstrate the **decomposition principle** for BRP coordination:

**Part A**: Solve the centralized portfolio problem (with buy/sell spread), extract dual variables (internal transfer prices), and show that sites solving independently at these prices reproduce the centralized optimum.

**Part B**: Implement iterative subgradient-based dual decomposition and discuss convergence.

The **buy/sell price spread** is what creates the coordination value: individually, sites pay the full spread; the BRP portfolio benefits from **internal netting** — surplus at one site offsets deficit at another, avoiding the spread.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pulp

np.random.seed(42)


## 1. Problem Data


In [ ]:
T = 24
hours = np.arange(T)
N = 5
eta_ch = 0.95; eta_dis = 0.95

site_params = [
    (5.0, 1.5, 10.0, 5.0),
    (8.0, 3.0, 15.0, 7.0),
    (3.0, 2.0, 5.0, 3.0),
    (12.0, 5.0, 20.0, 10.0),
    (6.0, 2.5, 8.0, 4.0),
]

pv_data, load_data = [], []
for i, (pp, lb, _, _) in enumerate(site_params):
    pv_data.append(np.maximum(0, pp*np.exp(-0.5*((hours-12+0.3*i)/3)**2) + 0.15*np.random.randn(T)))
    li = lb + 0.7*np.exp(-0.5*((hours-7.5)/2)**2) + 1.0*np.exp(-0.5*((hours-19)/2.5)**2) + 0.2*np.random.randn(T)
    load_data.append(np.maximum(li, 0.3))

# Buy/sell prices with 35% spread — creates the coordination value
price_base = np.maximum(45 + 20*np.sin(2*np.pi*(hours-6)/24) + 10*np.exp(-0.5*((hours-18)/3)**2), 15.0)
c_buy = price_base / 1000.0       # EUR/kWh
c_sell = price_base * 0.65 / 1000.0  # sell at 65% of buy

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].fill_between(hours, c_sell*1000, c_buy*1000, alpha=0.2, color='gray', label='Spread')
ax[0].plot(hours, c_buy*1000, 'r-', lw=2, label='Buy price')
ax[0].plot(hours, c_sell*1000, 'g-', lw=2, label='Sell price')
ax[0].set_xlabel('Hour'); ax[0].set_ylabel('EUR/MWh'); ax[0].set_title('Market Prices')
ax[0].legend(); ax[0].grid(alpha=0.3)

net = [load_data[i] - pv_data[i] for i in range(N)]
agg_net = sum(net)
for i in range(N):
    ax[1].plot(hours, net[i], '-', alpha=0.5, lw=1, label=f'Site {i}')
ax[1].plot(hours, agg_net, 'k-', lw=2.5, label='Aggregate')
ax[1].axhline(0, color='k', lw=0.5)
ax[1].set_xlabel('Hour'); ax[1].set_ylabel('kW')
ax[1].set_title('Net Load (Load - PV)'); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig('/tmp/nb4_data.png', dpi=100); plt.show()
print(f"Setup: {N} sites, {T} hours, spread = 35%")


## 2. Reference Solutions


In [ ]:
def solve_site_with_spread(pv, load, Em, Pm, c_b, c_s):
    """Solve single site LP with buy/sell spread."""
    T_loc = len(pv); Emin = Em*0.1; S0 = Em*0.5
    prob = pulp.LpProblem("Site", pulp.LpMinimize)
    p_ch = [pulp.LpVariable(f"c{t}", 0, Pm) for t in range(T_loc)]
    p_dis = [pulp.LpVariable(f"d{t}", 0, Pm) for t in range(T_loc)]
    p_buy = [pulp.LpVariable(f"b{t}", 0) for t in range(T_loc)]
    p_sell = [pulp.LpVariable(f"s{t}", 0) for t in range(T_loc)]
    soc = [pulp.LpVariable(f"e{t}", Emin, Em) for t in range(T_loc)]
    prob += pulp.lpSum([c_b[t]*p_buy[t] - c_s[t]*p_sell[t] for t in range(T_loc)])
    for t in range(T_loc):
        prob += pv[t] + p_dis[t] + p_buy[t] == load[t] + p_ch[t] + p_sell[t]
        sp = S0 if t==0 else soc[t-1]
        prob += soc[t] == sp + eta_ch*p_ch[t] - p_dis[t]/eta_dis
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    g = np.array([pulp.value(p_buy[t]) - pulp.value(p_sell[t]) for t in range(T_loc)])
    return g, pulp.value(prob.objective)

# A. Independent: each site faces buy/sell spread individually
g_indep = np.zeros((N, T))
indep_costs = []
for i in range(N):
    g_i, cost_i = solve_site_with_spread(pv_data[i], load_data[i], 
                                          site_params[i][2], site_params[i][3], c_buy, c_sell)
    g_indep[i] = g_i
    indep_costs.append(cost_i)
total_indep = sum(indep_costs)

# B. Centralized: portfolio faces buy/sell spread on aggregate
prob_c = pulp.LpProblem("Central", pulp.LpMinimize)
pc = {}; pd = {}; sc = {}
for i in range(N):
    Em, Pm = site_params[i][2], site_params[i][3]
    for t in range(T):
        pc[i,t] = pulp.LpVariable(f"c{i}_{t}", 0, Pm)
        pd[i,t] = pulp.LpVariable(f"d{i}_{t}", 0, Pm)
        sc[i,t] = pulp.LpVariable(f"e{i}_{t}", Em*0.1, Em)
pb = [pulp.LpVariable(f"pb_{t}", 0) for t in range(T)]
ps = [pulp.LpVariable(f"ps_{t}", 0) for t in range(T)]
prob_c += pulp.lpSum([c_buy[t]*pb[t] - c_sell[t]*ps[t] for t in range(T)])
for t in range(T):
    for i in range(N):
        Em = site_params[i][2]; sp = Em*0.5 if t==0 else sc[i,t-1]
        prob_c += sc[i,t] == sp + eta_ch*pc[i,t] - pd[i,t]/eta_dis
    prob_c += (pulp.lpSum([load_data[i][t] + pc[i,t] - pv_data[i][t] - pd[i,t]
               for i in range(N)]) == pb[t] - ps[t], f"PortBal_{t}")
prob_c.solve(pulp.PULP_CBC_CMD(msg=0))
central_cost = pulp.value(prob_c.objective)
z_central = np.array([pulp.value(pb[t]) - pulp.value(ps[t]) for t in range(T)])

# Extract dual variables of portfolio balance constraint
mu_star = np.array([prob_c.constraints[f"PortBal_{t}"].pi for t in range(T)])

print(f"Independent (sum of site costs):  {total_indep:.4f} EUR")
print(f"Centralized (portfolio cost):     {central_cost:.4f} EUR")
print(f"Coordination benefit:             {total_indep - central_cost:.4f} EUR ({100*(total_indep-central_cost)/max(abs(total_indep),0.01):.1f}%)")


## Part A: Optimal Internal Prices from Dual Variables

The dual variable $\mu^*_t$ of the portfolio balance constraint represents the **optimal internal transfer price**. It satisfies $c^{sell}_t \leq \mu^*_t \leq c^{buy}_t$.

When the portfolio is:
- **Net buying**: $\mu^*_t = c^{buy}_t$ (marginal cost = buy price)
- **Net selling**: $\mu^*_t = -c^{sell}_t$ (marginal value = sell price, sign convention)
- **Internally balanced**: $\mu^*_t$ lies strictly between (netting zone)


In [ ]:
# Solve each site independently at the optimal internal price
def solve_site_at_mu(pv, load, Em, Pm, mu):
    """Site LP: minimize mu^T * g subject to BESS constraints."""
    T_loc = len(pv); Emin = Em*0.1; S0 = Em*0.5
    prob = pulp.LpProblem("SiteMu", pulp.LpMinimize)
    p_ch = [pulp.LpVariable(f"c{t}", 0, Pm) for t in range(T_loc)]
    p_dis = [pulp.LpVariable(f"d{t}", 0, Pm) for t in range(T_loc)]
    p_buy = [pulp.LpVariable(f"b{t}", 0) for t in range(T_loc)]
    p_sell = [pulp.LpVariable(f"s{t}", 0) for t in range(T_loc)]
    soc = [pulp.LpVariable(f"e{t}", Emin, Em) for t in range(T_loc)]
    prob += pulp.lpSum([mu[t]*(p_buy[t] - p_sell[t]) for t in range(T_loc)])
    for t in range(T_loc):
        prob += pv[t] + p_dis[t] + p_buy[t] == load[t] + p_ch[t] + p_sell[t]
        sp = S0 if t==0 else soc[t-1]
        prob += soc[t] == sp + eta_ch*p_ch[t] - p_dis[t]/eta_dis
    prob.solve(pulp.PULP_CBC_CMD(msg=0))
    return np.array([pulp.value(p_buy[t]) - pulp.value(p_sell[t]) for t in range(T_loc)])

g_decomp = np.zeros((N, T))
for i in range(N):
    g_decomp[i] = solve_site_at_mu(pv_data[i], load_data[i],
                                    site_params[i][2], site_params[i][3], mu_star)
z_decomp = np.sum(g_decomp, axis=0)

# Evaluate portfolio cost of decomposed solution
decomp_port_cost = np.sum(c_buy * np.maximum(z_decomp, 0) - c_sell * np.maximum(-z_decomp, 0))

print(f"Centralized cost:                {central_cost:.4f} EUR")
print(f"Decomposed at μ* (portfolio):    {decomp_port_cost:.4f} EUR")
print(f"Gap:                             {abs(decomp_port_cost - central_cost):.6f} EUR")
print()
print(f"Optimal internal prices μ* (EUR/MWh):")
for t in [0, 6, 9, 12, 15, 18, 21]:
    in_netting = c_sell[t] < mu_star[t] < c_buy[t]
    zone = "NETTING" if in_netting else ("BUY" if mu_star[t] >= c_buy[t]-0.001 else "SELL")
    print(f"  h={t:2d}: μ*={mu_star[t]*1000:6.1f}, buy={c_buy[t]*1000:5.1f}, sell={c_sell[t]*1000:5.1f} [{zone}]")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Internal price vs buy/sell prices
axes[0,0].fill_between(hours, c_sell*1000, c_buy*1000, alpha=0.15, color='gray', label='Spread zone')
axes[0,0].plot(hours, c_buy*1000, 'r-', alpha=0.5, label='Buy')
axes[0,0].plot(hours, c_sell*1000, 'g-', alpha=0.5, label='Sell')
axes[0,0].plot(hours, mu_star*1000, 'b-o', ms=4, lw=2, label='Internal μ*')
axes[0,0].set_xlabel('Hour'); axes[0,0].set_ylabel('EUR/MWh')
axes[0,0].set_title('Optimal Internal Transfer Price')
axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Portfolio net position
axes[0,1].step(hours, z_central, 'r--', where='mid', lw=2, label='Centralized')
axes[0,1].step(hours, z_decomp, 'b-', where='mid', lw=2, alpha=0.7, label='Decomposed')
z_ind = np.sum(g_indep, axis=0)
axes[0,1].step(hours, z_ind, 'gray', where='mid', lw=1.5, ls=':', label='Independent')
axes[0,1].axhline(0, color='k', lw=0.5)
axes[0,1].set_xlabel('Hour'); axes[0,1].set_ylabel('kW')
axes[0,1].set_title('Portfolio Net Position'); axes[0,1].legend(); axes[0,1].grid(alpha=0.3)

# Per-site exchange (decomposed)
for i in range(N):
    axes[1,0].plot(hours, g_decomp[i], '-o', ms=2, label=f'Site {i}')
axes[1,0].plot(hours, z_decomp, 'k-', lw=2.5, label='Portfolio')
axes[1,0].axhline(0, color='k', lw=0.5)
axes[1,0].set_xlabel('Hour'); axes[1,0].set_ylabel('kW')
axes[1,0].set_title('Decomposed: Site Grid Exchanges'); axes[1,0].legend(fontsize=6)
axes[1,0].grid(alpha=0.3)

# Cost comparison bar chart
labels = ['Indep.\n(sum sites)', 'Centralized', 'Decomposed\nat μ*']
vals = [total_indep, central_cost, decomp_port_cost]
colors = ['gray', 'red', 'blue']
bars = axes[1,1].bar(labels, vals, color=colors, alpha=0.7)
axes[1,1].set_ylabel('EUR'); axes[1,1].set_title('Cost Comparison')
for bar, val in zip(bars, vals):
    axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                   f'{val:.3f}', ha='center', fontsize=9)
axes[1,1].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.savefig('/tmp/nb4_partA.png', dpi=100); plt.show()


## Part B: Iterative Dual Decomposition

In practice, we discover $\mu^*$ iteratively via **subgradient ascent** on the Lagrangian dual:

$$\mu^{k+1}_t = \mu^k_t + \alpha_k \cdot \left(z^k_t - \hat{z}_t\right)$$

where $z^k_t = \sum_i g_i^k(t)$ is the current portfolio position and $\hat{z}_t$ is the desired one.


In [ ]:
mu_iter = (c_buy + c_sell) / 2  # start at midpoint
max_iter = 80
g_iter = np.zeros((N, T))
cost_hist = []
best_cost = 1e10
best_g = None

for k in range(max_iter):
    alpha_k = 0.003 / (1 + 0.03*k)  # diminishing step
    
    for i in range(N):
        g_iter[i] = solve_site_at_mu(pv_data[i], load_data[i],
                                      site_params[i][2], site_params[i][3], mu_iter)
    z_k = np.sum(g_iter, axis=0)
    port_cost = np.sum(c_buy * np.maximum(z_k, 0) - c_sell * np.maximum(-z_k, 0))
    cost_hist.append(port_cost)
    
    if port_cost < best_cost:
        best_cost = port_cost
        best_g = g_iter.copy()
    
    # Subgradient: portfolio marginal cost w.r.t. z
    subgrad = np.where(z_k > 0, c_buy, -c_sell)
    mu_iter = mu_iter + alpha_k * (subgrad - mu_iter)
    mu_iter = np.clip(mu_iter, c_sell*0.5, c_buy*1.5)

print(f"After {max_iter} iterations:")
print(f"  Best portfolio cost:   {best_cost:.4f} EUR")
print(f"  Final iteration cost:  {cost_hist[-1]:.4f} EUR")
print(f"  Centralized optimum:   {central_cost:.4f} EUR")
print(f"  Independent baseline:  {total_indep:.4f} EUR")
print(f"  Improvement vs indep:  {total_indep - best_cost:.4f} EUR")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(cost_hist, 'b-', alpha=0.5)
running_best = np.minimum.accumulate(cost_hist)
axes[0].plot(running_best, 'b-', lw=2, label='Best so far')
axes[0].axhline(central_cost, color='r', ls='--', label=f'Central = {central_cost:.3f}')
axes[0].axhline(total_indep, color='gray', ls=':', label=f'Independent = {total_indep:.3f}')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('EUR')
axes[0].set_title('Dual Decomposition Convergence'); axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)

labels = ['Independent', 'Iterative\n(best)', 'Centralized']
vals = [total_indep, best_cost, central_cost]
colors = ['gray', 'steelblue', 'red']
axes[1].bar(labels, vals, color=colors, alpha=0.7)
for j, v in enumerate(vals):
    axes[1].text(j, v+0.01, f'{v:.3f}', ha='center', fontsize=10)
axes[1].set_ylabel('EUR'); axes[1].set_title('Final Comparison'); axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig('/tmp/nb4_iter.png', dpi=100); plt.show()


## 6. Key Insights for the Thesis

1. **Decomposition principle works**: at the optimal internal price $\mu^*$ (from the centralized dual), sites independently reproduce the centralized solution. This proves that **price-based coordination** can achieve portfolio-level optimality.

2. **Internal transfer price interpretation**: $\mu^*_t$ lies in $[c^{sell}_t, c^{buy}_t]$. When the portfolio is net-buying at hour $t$, $\mu^*_t \approx c^{buy}_t$; when net-selling, $\mu^*_t \approx c^{sell}_t$. Within the "netting zone" the price is interior — this is where the coordination benefit comes from.

3. **Iterative discovery of $\mu^*$** via subgradient methods is possible but challenging for LP sub-problems due to vertex-hopping. ADMM with quadratic proximal regularization (Boyd et al., §3) or Benders decomposition can improve convergence.

4. **Privacy**: sites only share net grid exchange — all internal data stays private (GDPR compliance). The coordinator only sends the price signal.

5. **Practical recommendation**: for the company's scale (<100 sites), use centralized MILP (Notebook 2) and extract $\mu^*$ as internal transfer prices for benefit sharing. Deploy decomposition only if:
   - Number of sites exceeds solver scalability limits
   - Prosumers demand data privacy guarantees
   - Sites need local autonomy (e.g., backup SoC constraints)

6. **Hybrid approach**: solve centralized problem offline (e.g., daily), extract $\mu^*$, then let each site's MPC use $\mu^*$ as the price signal for real-time dispatch. This combines centralized planning optimality with distributed real-time execution.

---

*Next: Notebook 5 implements MPC for real-time balancing.*
